# Regression Trees Example: Predicting Running Back Yards per Carry

The aim of this notebook is to show a real use of the regression tree class defined in our ML package.

Our analysis aims to use Division 1 College Football player athleticism data to predict a running back's yards per carry (YPC). This athleticism data includes speed, accleration, and change of direction metrics in the form of counts, averages, and percentiles.

We hypothesize that running backs who have higher athleticism scores in speed, acceleration, and change of direction will have a higher YPC. However, we see some limitations with this apporach as running back efficiency is likely determined by external team factors and the skills a running back has.

In [25]:
# Import necessary libraries
import sys
import os

# Send Python to the project root so we can import our library
project_root = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/src"
sys.path.append(project_root)

# Import our ML library and other necessary libraries
import rice_ml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [26]:
# Load the athleticism data
ath_data_path = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/data/college_RB_athleticism.csv"
ath_data = pd.read_csv(ath_data_path)

# Load the running back stats data
stats_data_path = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/data/college_RB_stats.csv"
stats_data = pd.read_csv(stats_data_path)

Now that we have the data loaded we can clean it and set up for our model. The data is already filtered to just be running backs.

In [27]:
# Merge stats_data and ath_data on 'Player Id' and 'Name'
merged_data = pd.merge(stats_data, ath_data, on=['Player Id', 'Name'], how='inner')

# Select features and target variable
features = merged_data[["Player Id", "Name", "ATT", "ATH PER", 
                        "MAX SPD", "AVG MAX", "SPD PER", "20+ MPH", 
                        "15+ MPH", "ACC PER", "COD PER", "YPC"]]

# Minimum attempts threshold
min_attempts = 20
filtered_data = features[features["ATT"] >= min_attempts]

# Drop attempts column
filtered_data = filtered_data.drop(columns=["ATT"])

# View the first few rows of the filtered data
print(filtered_data.head())

   Player Id                Name  ATH PER  MAX SPD  AVG MAX  SPD PER  20+ MPH  \
0      70767        Damien Moore      8.2     15.9     14.7     17.1        0   
1      72376           Cam Davis     63.4     18.1     17.6     35.2        0   
2      73289     Ikaika Ragsdale     61.7     18.1     16.6     69.4        0   
3      73621      Elijah Gilliam     70.4     19.9     18.6     70.5        0   
6      86353  Byron Cardwell Jr.     87.0     20.8     19.1     90.4        1   

   15+ MPH  ACC PER  COD PER  YPC  
0        2     11.1     11.8  2.8  
1       10     66.2     75.9  3.4  
2       10     60.4     43.8  3.5  
3        9     74.0     66.0  5.3  
6       14     83.7     69.9  5.5  


We first merged the athletcism and stats data frames into one set that has player identification, atheliticism metrics, rushing attempts (ATT), and Yards per Carry. 

We use ATT to filter out very low volume running backs who likely have high YPC volatility.

We will now use Principal Component Analysis on the athleticism features to overcome the inevitable multicollinearity between the eight athleticism features.

In [31]:
# Use our library to perform PCA on the athleticism features
# Shrink to 2 variables for easy visualization
# Here are the athleticism features we want to include in the PCA:
athleticism_features = ["ATH PER", "MAX SPD", "AVG MAX", "SPD PER", 
                        "20+ MPH", "15+ MPH", "ACC PER", "COD PER"]

# Scale features before PCA
scaler = rice_ml.StandardScaler()
X_scaled = scaler.fit_transform(filtered_data[athleticism_features])

# Perform PCA using our library
pca = rice_ml.PCA(n_components=4)
pca_result = pca.fit_transform(X_scaled)

# Print the explained variance ratio to see how much variance is captured by the principal components
print(f"Explained Variance Ratio: {pca.explained_variance_ratio}")
print(f"Total Variance Captured: {sum(pca.explained_variance_ratio):.2%}")

# Create a new DataFrame to hold the PCA results and the target variable (YPC)
pca_df = pd.DataFrame(data=pca_result, columns=['PC1', 'PC2', 'PC3', 'PC4'])
pca_df['YPC'] = filtered_data['YPC'].values



Explained Variance Ratio: [0.6403828  0.12663903 0.09141767 0.06094973]
Total Variance Captured: 91.94%


To successfully integrate PCA in our analysis, we first needed to standardize the eight features. By doing so, all of the features are treated equally and understood in the same way by PCA. This overcomes the different types of values we have (percentiles, counts, averages).

From our PCA results, we decided to keep four principal components that account for 91.94% of the variance. Each component accounts for more than 5% of the total variance, giving us confidence to use all four of the top components.

Now we have our four features and updated data frame that is ready for model training. 

In [32]:
# Use 80/20 train-test split
pca_df = pca_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(0.8 * len(pca_df))
train_data = pca_df.iloc[:train_size]
test_data = pca_df.iloc[train_size:]

# Train regression tree model using our library
model = rice_ml.RegressionTree()
X_train = train_data[["PC1", "PC2", "PC3", "PC4"]].values
y_train = train_data["YPC"].values
model.train(X_train, y_train)


Now that we have trained the model, we can create predictions and evaluate our model's performance.

In [33]:
# Make predictions on the test set
predictions = model.predict(test_data[["PC1", "PC2", "PC3", "PC4"]].values)

# Evaluate the model using measurements (MAE and R^2) from our library
mae = rice_ml.mean_absolute_error(test_data["YPC"].values, predictions)
r2 = rice_ml.r_squared_score(test_data["YPC"].values, predictions)

# Print the evaluation results
print(f"Mean Absolute Error: {mae}")
print(f"R^2 Score: {r2}")

Mean Absolute Error: 0.8445644197361261
R^2 Score: 0.16809234997389544
